In [1]:
from utils import *
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

## Linear Regression

Simple population-level linear regression model. The country is not considered.

In [10]:
X, Y, C = load_height_data()
model = LinearRegression().fit(X, Y)
y_pred = model.predict(X)
print("Loss:", np.mean((y_pred - Y)**2))

Loss: 45.53098


Next, we try learning a separate linear regression model for each country. Notice the loss is much better.

In [2]:
X, Y, C = load_height_data()
model = ContextLinearRegression().fit(X, Y, C)
y_pred = model.predict(X, C)
print("Loss:", np.mean((y_pred - Y)**2))

Loss: 23.753902446756058


## Multilayer Perceptron
The MLP without country as input performs about equally well as learning a separate linear regression model for each country.

In [7]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = "cpu"
print(f"Using {device} device")

Using mps device


In [8]:
model = NeuralNetwork().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y)
    test_loss /= num_batches
    print(f"Test Error: \n Avg loss: {test_loss:>8f} \n")

In [9]:
full_dataset = HeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=500)
test_dataloader = DataLoader(test_data, batch_size=500)

epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 407.988770  [  500/168000]
loss: 37.909214  [50500/168000]
loss: 29.065962  [100500/168000]
loss: 30.781721  [150500/168000]
Test Error: 
 Avg loss: 29.782465 

Epoch 2
-------------------------------
loss: 30.127571  [  500/168000]
loss: 30.403055  [50500/168000]
loss: 24.708633  [100500/168000]
loss: 26.509258  [150500/168000]
Test Error: 
 Avg loss: 26.287083 

Epoch 3
-------------------------------
loss: 26.991072  [  500/168000]
loss: 27.215794  [50500/168000]
loss: 23.519459  [100500/168000]
loss: 24.660925  [150500/168000]
Test Error: 
 Avg loss: 25.153011 

Epoch 4
-------------------------------
loss: 26.336369  [  500/168000]
loss: 25.783779  [50500/168000]
loss: 22.895889  [100500/168000]
loss: 23.824352  [150500/168000]
Test Error: 
 Avg loss: 24.894924 

Epoch 5
-------------------------------
loss: 25.911299  [  500/168000]
loss: 24.984848  [50500/168000]
loss: 22.518305  [100500/168000]
loss: 23.457809  [150500/168000]
Test 

## Context-Sensitive Network
The task is one-hot-encoded and appended to the input.